# EEG-to-Language Model: CSBrain Encoder + TinyLlama Decoder

This notebook implements a multimodal EEG-to-Language architecture (inspired by LLaVA) where:
- **Encoder**: CSBrain (frozen EEG foundation model) extracts EEG representations
- **Projection**: A 2-layer MLP maps EEG tokens to the LLM embedding space
- **Decoder**: TinyLlama-1.1B-Chat (4-bit quantized + LoRA) generates free-form text describing the EEG

We use the **BCIC-IV-2a** motor imagery dataset (4 classes) as a proof-of-concept.

```
EEG (batch, 22, 4, 200)
  → CSBrain (frozen) → (batch, 22, 4, 200)
  → Region pooling (3 regions) → (batch, 12, 200)
  → MLP projection → (batch, 12, 2048)
  → Concat with text prompt embeddings
  → TinyLlama-1.1B-Chat (4-bit, LoRA) → generated reasoning text
```

In [1]:
import sys
import os
import ssl
import argparse
import urllib.request
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from timeit import default_timer as timer
import scipy.io
import lmdb
import pickle
from scipy.signal import butter, lfilter, resample
import copy

ssl._create_default_https_context = ssl._create_unverified_context

# Set seed for reproducibility
def setup_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(42)

## Configuration

In [2]:
# Paths
CSBRAIN_WEIGHTS_PATH = "pth/CSBrain.pth"  # Pretrained CSBrain encoder weights
FINETUNED_WEIGHTS_PATH = "model_weights/epoch5_acc_0.57726_kappa_0.43634_f1_0.56665.pth"  # Optional: fine-tuned BCIC weights
RAW_DATA_DIR = "data/BCICIV2a/raw"
LMDB_DIR = "data/BCICIV2a/processed_lmdb"
SAVE_DIR = "pth_downtasks/eeg_llm_bcic"

# LLM config
LLM_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Open-source, no auth required
LLM_DIM = 2048  # Hidden dim of TinyLlama-1.1B
LORA_RANK = 8
LORA_ALPHA = 16

# Training config
NUM_CLASSES = 4
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8  # Effective batch = 4 * 8 = 32
EPOCHS = 20
WARMUP_EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 0.01
MAX_TARGET_LEN = 128
DROPOUT = 0.1

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# BCIC-IV-2a class labels
CLASS_NAMES = {0: "Left Hand", 1: "Right Hand", 2: "Both Feet", 3: "Tongue"}

Using device: cuda:0


## 1. Download & Preprocess BCIC-IV-2a Dataset

Downloads the 9 subjects from BNCI Horizon 2020, applies bandpass filter, crops motor imagery window, resamples, and stores in LMDB.

In [3]:
os.makedirs(RAW_DATA_DIR, exist_ok=True)

BASE_URL = "http://bnci-horizon-2020.eu/database/data-sets/001-2014"
SUBJECTS = [f"A{i:02d}" for i in range(1, 10)]
SUFFIXES = ["T", "E"]

for subj in SUBJECTS:
    for suffix in SUFFIXES:
        fname = f"{subj}{suffix}.mat"
        fpath = os.path.join(RAW_DATA_DIR, fname)
        if os.path.exists(fpath):
            print(f"Already exists: {fname}")
            continue
        url = f"{BASE_URL}/{fname}"
        print(f"Downloading {fname}...")
        try:
            urllib.request.urlretrieve(url, fpath)
            print(f"  Saved to {fpath}")
        except Exception as e:
            print(f"  Failed: {e}")

print(f"\nFiles in {RAW_DATA_DIR}: {sorted(os.listdir(RAW_DATA_DIR))}")

Already exists: A01T.mat
Already exists: A01E.mat
Already exists: A02T.mat
Already exists: A02E.mat
Already exists: A03T.mat
Already exists: A03E.mat
Already exists: A04T.mat
Already exists: A04E.mat
Already exists: A05T.mat
Already exists: A05E.mat
Already exists: A06T.mat
Already exists: A06E.mat
Already exists: A07T.mat
Already exists: A07E.mat
Already exists: A08T.mat
Already exists: A08E.mat
Already exists: A09T.mat
Already exists: A09E.mat

Files in data/BCICIV2a/raw: ['A01E.mat', 'A01T.mat', 'A02E.mat', 'A02T.mat', 'A03E.mat', 'A03T.mat', 'A04E.mat', 'A04T.mat', 'A05E.mat', 'A05T.mat', 'A06E.mat', 'A06T.mat', 'A07E.mat', 'A07T.mat', 'A08E.mat', 'A08T.mat', 'A09E.mat', 'A09T.mat']


In [4]:
def butter_bandpass(low_cut, high_cut, fs, order=5):
    nyq = 0.5 * fs
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype="band")
    return b, a

if os.path.exists(os.path.join(LMDB_DIR, "data.mdb")):
    print(f"LMDB already exists at {LMDB_DIR}, skipping preprocessing.")
else:
    os.makedirs(LMDB_DIR, exist_ok=True)

    files_dict = {
        "train": ["A01E.mat", "A01T.mat", "A02E.mat", "A02T.mat", "A03E.mat", "A03T.mat",
                  "A04E.mat", "A04T.mat", "A05E.mat", "A05T.mat"],
        "val": ["A06E.mat", "A06T.mat", "A07E.mat", "A07T.mat"],
        "test": ["A08E.mat", "A08T.mat", "A09E.mat", "A09T.mat"],
    }

    dataset_keys = {"train": [], "val": [], "test": []}
    db = lmdb.open(LMDB_DIR, map_size=1610612736)

    for split in files_dict:
        for file in files_dict[split]:
            fpath = os.path.join(RAW_DATA_DIR, file)
            if not os.path.exists(fpath):
                print(f"WARNING: {fpath} not found, skipping.")
                continue
            print(f"Processing {file} ({split})...")
            data = scipy.io.loadmat(fpath)
            num = len(data["data"][0])

            for j in range(3, num):
                raw_data = data["data"][0, j][0, 0][0][:, :22]
                events = data["data"][0, j][0, 0][1][:, 0]
                labels = data["data"][0, j][0, 0][2][:, 0]
                length = raw_data.shape[0]
                events = events.tolist()
                events.append(length)

                annos = []
                for i in range(len(events) - 1):
                    annos.append((events[i], events[i + 1]))

                for i, (anno, label) in enumerate(zip(annos, labels)):
                    sample = raw_data[anno[0]:anno[1]].transpose(1, 0)
                    sample = sample - np.mean(sample, axis=0, keepdims=True)
                    b, a = butter_bandpass(0.3, 50, 250)
                    sample = lfilter(b, a, sample, -1)
                    sample = sample[:, 2 * 250:6 * 250]
                    sample = resample(sample, 800, axis=-1)
                    sample = sample.reshape(22, 4, 200)

                    sample_key = f"{file[:-4]}-{j}-{i}"
                    data_dict = {"sample": sample, "label": label - 1}
                    txn = db.begin(write=True)
                    txn.put(key=sample_key.encode(), value=pickle.dumps(data_dict))
                    txn.commit()
                    dataset_keys[split].append(sample_key)

    txn = db.begin(write=True)
    txn.put(key="__keys__".encode(), value=pickle.dumps(dataset_keys))
    txn.commit()
    db.close()

    for split in dataset_keys:
        print(f"  {split}: {len(dataset_keys[split])} samples")
    print("Preprocessing complete!")

LMDB already exists at data/BCICIV2a/processed_lmdb, skipping preprocessing.


## 2. Define Label-to-Text Mapping

Each motor imagery class maps to descriptive neuroscience text. Multiple paraphrases per class serve as training augmentation.

In [5]:
BCIC_LABEL_MAP = {
    0: [
        "The EEG recording shows motor imagery patterns for left hand movement. There is a clear event-related desynchronization (ERD) in the mu and beta bands over the right sensorimotor cortex (C4), indicating contralateral motor planning for left hand imagery.",
        "This EEG signal reveals left hand motor imagery. The neural patterns demonstrate suppressed mu rhythm activity over the right central region, consistent with imagined left hand movement and contralateral cortical activation.",
        "Analysis of the EEG indicates left hand motor imagery. Characteristic right-lateralized sensorimotor desynchronization in the 8-30 Hz range suggests the subject is imagining left hand movement.",
    ],
    1: [
        "The EEG recording shows motor imagery patterns for right hand movement. There is a clear event-related desynchronization (ERD) in the mu and beta bands over the left sensorimotor cortex (C3), indicating contralateral motor planning for right hand imagery.",
        "This EEG signal reveals right hand motor imagery. The neural patterns demonstrate suppressed mu rhythm activity over the left central region, consistent with imagined right hand movement and contralateral cortical activation.",
        "Analysis of the EEG indicates right hand motor imagery. Characteristic left-lateralized sensorimotor desynchronization in the 8-30 Hz range suggests the subject is imagining right hand movement.",
    ],
    2: [
        "The EEG recording shows motor imagery patterns for both feet movement. There is bilateral event-related desynchronization (ERD) over the central midline area (Cz), consistent with supplementary motor area activation for lower limb imagery.",
        "This EEG signal reveals feet motor imagery. The neural patterns demonstrate suppressed mu and beta rhythms over the vertex (Cz) and surrounding central electrodes, indicating imagined bilateral foot movement.",
        "Analysis of the EEG indicates both feet motor imagery. Midline central desynchronization in the sensorimotor frequency bands suggests the subject is imagining foot movement, with activation centered over the supplementary motor area.",
    ],
    3: [
        "The EEG recording shows motor imagery patterns for tongue movement. There is event-related desynchronization (ERD) over the lateral sensorimotor regions bilaterally, consistent with orofacial motor imagery involving the tongue area of the motor homunculus.",
        "This EEG signal reveals tongue motor imagery. The neural patterns demonstrate bilateral sensorimotor desynchronization with a distinct spatial pattern from limb imagery, indicating activation of the face and tongue representation in the motor cortex.",
        "Analysis of the EEG indicates tongue motor imagery. Widespread sensorimotor desynchronization with bilateral distribution suggests the subject is imagining tongue movement, activating the lateral portions of the primary motor cortex.",
    ],
}

# Keywords for evaluation
EMOTION_KEYWORDS = {
    0: ["left hand", "left-lateralized", "right sensorimotor", "right central", "contralateral"],
    1: ["right hand", "right-lateralized", "left sensorimotor", "left central", "contralateral"],
    2: ["feet", "foot", "bilateral", "midline", "cz", "vertex", "lower limb", "supplementary motor"],
    3: ["tongue", "orofacial", "face", "lateral portions"],
}

SYSTEM_PROMPT = "You are an expert EEG analyst specializing in brain-computer interfaces and motor imagery decoding. Analyze the provided EEG recording and describe the motor imagery task being performed."
USER_PROMPT = "Analyze this EEG recording and describe the motor imagery task the subject is performing, including the neural patterns observed."

print(f"Defined {len(BCIC_LABEL_MAP)} classes with {len(BCIC_LABEL_MAP[0])} paraphrases each")

Defined 4 classes with 3 paraphrases each


## 3. Dataset & Collator

In [6]:
def to_tensor(array):
    return torch.from_numpy(array).float()


class BCICLLMDataset(Dataset):
    """LMDB-based BCIC-IV-2a dataset returning (eeg_data, label_id)."""
    def __init__(self, data_dir, mode='train'):
        super().__init__()
        self.db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)
        with self.db.begin(write=False) as txn:
            self.keys = pickle.loads(txn.get('__keys__'.encode()))[mode]
        self.mode = mode

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        with self.db.begin(write=False) as txn:
            pair = pickle.loads(txn.get(key.encode()))
        data = pair['sample']
        label = int(pair['label'])
        return data / 100, label


class BCICLLMCollator:
    """Builds tokenized prompts and targets for the EEG-LLM model."""
    def __init__(self, tokenizer, max_target_len=128, mode='train'):
        self.tokenizer = tokenizer
        self.max_target_len = max_target_len
        self.mode = mode

        # TinyLlama chat template (Zephyr format)
        self.prompt_text = (
            f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
            f"<|user|>\n[EEG_TOKENS]\n{USER_PROMPT}</s>\n"
            f"<|assistant|>\n"
        )

        prompt_encoded = self.tokenizer(
            self.prompt_text, return_tensors="pt",
            add_special_tokens=False, padding=False
        )
        self.prompt_ids = prompt_encoded['input_ids'].squeeze(0)
        self.prompt_mask = prompt_encoded['attention_mask'].squeeze(0)

    def __call__(self, batch):
        eeg_data = np.array([x[0] for x in batch])
        labels = [x[1] for x in batch]
        batch_size = len(batch)

        target_texts = []
        for label_id in labels:
            paraphrases = BCIC_LABEL_MAP[label_id]
            if self.mode == 'train':
                text = random.choice(paraphrases)
            else:
                text = paraphrases[0]
            target_texts.append(text + "</s>")

        target_encoded = self.tokenizer(
            target_texts, return_tensors="pt",
            add_special_tokens=False, padding=True,
            truncation=True, max_length=self.max_target_len,
        )
        target_ids = target_encoded['input_ids']
        target_mask = target_encoded['attention_mask']

        prompt_ids = self.prompt_ids.unsqueeze(0).expand(batch_size, -1)
        prompt_mask = self.prompt_mask.unsqueeze(0).expand(batch_size, -1)

        return {
            'eeg_data': to_tensor(eeg_data),
            'prompt_ids': prompt_ids,
            'prompt_mask': prompt_mask,
            'target_ids': target_ids,
            'target_mask': target_mask,
            'label_ids': torch.tensor(labels, dtype=torch.long),
        }

## 4. Define the EEG-Language Model

Architecture:
- **CSBrain encoder** (frozen): `(batch, 22, 4, 200)` → `(batch, 22, 4, 200)`
- **EEGTokenReducer**: Pools channels by brain region (3 regions) → `(batch, 12, 200)` (only 12 tokens!)
- **EEGProjection**: MLP `200 → 2048 → 2048`
- **TinyLlama-1.1B-Chat**: 4-bit quantized with LoRA on q_proj, v_proj

In [7]:
from models.CSBrain import CSBrain


class EEGTokenReducer(nn.Module):
    """Pools channels within brain regions to reduce token count."""
    def __init__(self, area_config, temporal_pool_stride=1):
        super().__init__()
        self.area_config = area_config
        self.temporal_pool_stride = temporal_pool_stride

    def forward(self, x):
        batch, n_ch, n_patches, d_model = x.shape
        region_tokens = []
        for region_name in sorted(self.area_config.keys()):
            s = self.area_config[region_name]['slice']
            region_avg = x[:, s, :, :].mean(dim=1)
            region_tokens.append(region_avg)
        n_regions = len(region_tokens)
        tokens = torch.stack(region_tokens, dim=1)
        if self.temporal_pool_stride > 1:
            n_out = n_patches // self.temporal_pool_stride
            tokens = tokens[:, :, :n_out * self.temporal_pool_stride, :]
            tokens = tokens.view(batch, n_regions, n_out, self.temporal_pool_stride, d_model)
            tokens = tokens.mean(dim=3)
        return tokens.view(batch, -1, d_model)


class EEGProjection(nn.Module):
    """Projects EEG tokens to LLM hidden dim via 2-layer MLP."""
    def __init__(self, eeg_dim=200, llm_dim=2048, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(eeg_dim, llm_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, x):
        return self.proj(x)


class EEGLanguageModel(nn.Module):
    """CSBrain encoder + projection + TinyLlama decoder with LoRA."""

    def __init__(self, llm_model_name, csbrain_weights_path, llm_dim=2048,
                 lora_rank=8, lora_alpha=16, dropout=0.1, device='cuda:0'):
        super().__init__()
        self._device = torch.device(device)  # store target device

        # ---- BCIC-IV-2a brain region config ----
        bci42a_brain_regions = [
            0,
            4, 4, 4, 4, 4,
            4, 4, 4, 4, 4, 4, 4,
            4, 4, 4, 4, 4,
            1, 1, 1, 1,
        ]
        bci42a_electrode_labels = [
            "Fz",
            "FC3", "FC1", "FCZ", "FC2", "FC4",
            "C5", "C3", "C1", "CZ", "C2", "C4", "C6",
            "CP3", "CP1", "CPZ", "CP2", "CP4",
            "P1", "PZ", "P2", "POZ"
        ]
        bci42a_topology = {
            0: ["Fz"],
            4: ["FC3", "FC1", "FCZ", "FC2", "FC4", "C5", "C3", "C1", "CZ", "C2", "C4", "C6",
                "CP3", "CP1", "CPZ", "CP2", "CP4"],
            1: ["P1", "PZ", "P2", "POZ"]
        }
        region_groups = {}
        for i, region in enumerate(bci42a_brain_regions):
            region_groups.setdefault(region, []).append((i, bci42a_electrode_labels[i]))
        sorted_indices = []
        for region in sorted(region_groups.keys()):
            sorted_electrodes = sorted(region_groups[region],
                                       key=lambda x: bci42a_topology[region].index(x[1]))
            sorted_indices.extend([e[0] for e in sorted_electrodes])
        print(f"BCIC-IV-2a Sorted Indices: {sorted_indices}")

        # ---- 1. CSBrain Encoder (frozen, float32) ----
        self.eeg_encoder = CSBrain(
            in_dim=200, out_dim=200, d_model=200,
            dim_feedforward=800, seq_len=30,
            n_layer=12, nhead=8,
            brain_regions=bci42a_brain_regions,
            sorted_indices=sorted_indices
        )
        if csbrain_weights_path and os.path.exists(csbrain_weights_path):
            state_dict = torch.load(csbrain_weights_path, map_location='cpu')
            new_sd = {k.replace("module.", ""): v for k, v in state_dict.items()}
            msd = self.eeg_encoder.state_dict()
            msd.update({k: v for k, v in new_sd.items() if k in msd and v.size() == msd[k].size()})
            self.eeg_encoder.load_state_dict(msd)
            print(f"Loaded pretrained CSBrain weights")
        else:
            print("WARNING: No pretrained CSBrain weights found, using random init")
        self.eeg_encoder.proj_out = nn.Identity()
        for p in self.eeg_encoder.parameters():
            p.requires_grad = False

        # *** Move encoder to CUDA (device_map="auto" won't do this for us) ***
        self.eeg_encoder = self.eeg_encoder.to(self._device)

        # ---- 2. Token Reducer (no parameters) ----
        self.token_reducer = EEGTokenReducer(
            area_config=self.eeg_encoder.area_config,
            temporal_pool_stride=1
        )

        # ---- 3. Projection MLP — also moved to CUDA ----
        self.eeg_projection = EEGProjection(
            eeg_dim=200, llm_dim=llm_dim, dropout=dropout
        ).to(self._device)

        # ---- 4. TinyLlama Decoder (4-bit + LoRA) ----
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        print(f"Loading {llm_model_name}...")
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        self.llm = prepare_model_for_kbit_training(self.llm)
        lora_config = LoraConfig(
            r=lora_rank, lora_alpha=lora_alpha,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        )
        self.llm = get_peft_model(self.llm, lora_config)
        self.llm.print_trainable_parameters()
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def get_text_embeddings(self, token_ids):
        return self.llm.get_input_embeddings()(token_ids)

    def forward(self, eeg_data, prompt_ids, prompt_mask, target_ids, target_mask):
        batch_size = eeg_data.shape[0]
        # Encoder runs float32 on CUDA; autocast disabled here to avoid dtype mismatch
        with torch.no_grad():
            self.eeg_encoder.eval()
            with torch.amp.autocast('cuda', enabled=False):
                eeg_features = self.eeg_encoder(eeg_data.float())
        eeg_tokens = self.token_reducer(eeg_features)
        eeg_embeds = self.eeg_projection(eeg_tokens.to(self.eeg_projection.proj[0].weight.dtype))

        prompt_embeds = self.get_text_embeddings(prompt_ids)
        target_embeds = self.get_text_embeddings(target_ids)
        inputs_embeds = torch.cat([prompt_embeds, eeg_embeds, target_embeds], dim=1)

        eeg_attn_mask = torch.ones(batch_size, eeg_tokens.shape[1],
                                   device=eeg_data.device, dtype=prompt_mask.dtype)
        attention_mask = torch.cat([prompt_mask, eeg_attn_mask, target_mask], dim=1)

        ignore_len = prompt_ids.shape[1] + eeg_tokens.shape[1]
        ignore_labels = torch.full((batch_size, ignore_len), -100,
                                   device=eeg_data.device, dtype=target_ids.dtype)
        labels = torch.cat([ignore_labels, target_ids], dim=1)
        labels = labels.masked_fill(labels == self.tokenizer.pad_token_id, -100)

        return self.llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)

    @torch.no_grad()
    def generate(self, eeg_data, prompt_ids, prompt_mask, max_new_tokens=128):
        self.eeg_encoder.eval()
        with torch.amp.autocast('cuda', enabled=False):
            eeg_features = self.eeg_encoder(eeg_data.float())
        eeg_tokens = self.token_reducer(eeg_features)
        eeg_embeds = self.eeg_projection(eeg_tokens.to(self.eeg_projection.proj[0].weight.dtype))

        prompt_embeds = self.get_text_embeddings(prompt_ids)
        inputs_embeds = torch.cat([prompt_embeds, eeg_embeds], dim=1)
        eeg_attn_mask = torch.ones(eeg_data.shape[0], eeg_tokens.shape[1],
                                   device=eeg_data.device, dtype=prompt_mask.dtype)
        attention_mask = torch.cat([prompt_mask, eeg_attn_mask], dim=1)

        outputs = self.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)


## 5. Build Model & Data Loaders

In [8]:
# Build model
model = EEGLanguageModel(
    llm_model_name=LLM_MODEL_NAME,
    csbrain_weights_path=CSBRAIN_WEIGHTS_PATH,
    llm_dim=LLM_DIM,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    dropout=DROPOUT,
    device=str(DEVICE),
)

tokenizer = model.tokenizer
print(f"\nModel built. Tokenizer vocab size: {len(tokenizer)}")

BCIC-IV-2a Sorted Indices: [0, 18, 19, 20, 21, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Loaded pretrained CSBrain weights


c:\Users\manoj\Projects\Mtech_Project2\CSBrain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:02<00:00, 83.49it/s, Materializing param=model.norm.weight]                               


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023

Model built. Tokenizer vocab size: 32000


In [9]:
# Build data loaders
train_set = BCICLLMDataset(LMDB_DIR, mode='train')
val_set = BCICLLMDataset(LMDB_DIR, mode='val')
test_set = BCICLLMDataset(LMDB_DIR, mode='test')

train_collator = BCICLLMCollator(tokenizer, MAX_TARGET_LEN, mode='train')
eval_collator = BCICLLMCollator(tokenizer, MAX_TARGET_LEN, mode='eval')

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, collate_fn=train_collator, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, collate_fn=eval_collator, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, collate_fn=eval_collator, shuffle=False)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

# Quick shape check
sample_batch = next(iter(train_loader))
print(f"EEG data shape: {sample_batch['eeg_data'].shape}")
print(f"Prompt IDs shape: {sample_batch['prompt_ids'].shape}")
print(f"Target IDs shape: {sample_batch['target_ids'].shape}")
print(f"Label IDs: {sample_batch['label_ids']}")

Train: 2784, Val: 1152, Test: 1152
EEG data shape: torch.Size([4, 22, 4, 200])
Prompt IDs shape: torch.Size([4, 101])
Target IDs shape: torch.Size([4, 61])
Label IDs: tensor([0, 3, 1, 2])


## 6. Training

Two-phase training strategy:
1. **Phase 1 (Warmup)**: Train only the projection MLP with higher LR — aligns EEG embedding space to LLM space
2. **Phase 2 (Joint)**: Train projection + LoRA adapters together with lower LR

In [10]:
def extract_label(text, keywords_map):
    """Extract predicted label from generated text via keyword matching."""
    text_lower = text.lower()
    best_label = -1
    best_count = 0
    for label_id, keywords in keywords_map.items():
        count = sum(1 for kw in keywords if kw in text_lower)
        if count > best_count:
            best_count = count
            best_label = label_id
    return best_label


@torch.no_grad()
def evaluate(model, data_loader, desc="Evaluating", max_new_tokens=32):
    """Generate text and compute accuracy via keyword extraction.

    max_new_tokens=32 is enough for keyword matching and avoids OOM on 8GB VRAM.
    """
    model.eval()
    torch.cuda.empty_cache()  # Free fragmented VRAM before generation
    correct = 0
    total = 0
    for batch in tqdm(data_loader, desc=desc, mininterval=10):
        generated_texts = model.generate(
            eeg_data=batch['eeg_data'].cuda(),
            prompt_ids=batch['prompt_ids'].cuda(),
            prompt_mask=batch['prompt_mask'].cuda(),
            max_new_tokens=max_new_tokens,
        )
        label_ids = batch['label_ids'].numpy()
        for text, true_label in zip(generated_texts, label_ids):
            pred_label = extract_label(text, EMOTION_KEYWORDS)
            if pred_label == true_label:
                correct += 1
            total += 1
    acc = correct / total if total > 0 else 0
    return acc, correct, total


def train_phase(model, train_loader, val_loader, trainable_params, epochs, lr,
                phase_name, grad_accum_steps, best_state):
    """Train for a given phase."""
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader), eta_min=1e-6
    )
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        model.eeg_encoder.eval()
        losses = []
        start_time = timer()
        optimizer.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f"{phase_name} Epoch {epoch+1}/{epochs}", mininterval=10)):
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(
                    eeg_data=batch['eeg_data'].cuda(),
                    prompt_ids=batch['prompt_ids'].cuda(),
                    prompt_mask=batch['prompt_mask'].cuda(),
                    target_ids=batch['target_ids'].cuda(),
                    target_mask=batch['target_mask'].cuda(),
                )
                loss = outputs.loss / grad_accum_steps

            scaler.scale(loss).backward()
            losses.append(loss.item() * grad_accum_steps)

            if (step + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            scheduler.step()

        elapsed = (timer() - start_time) / 60
        avg_loss = np.mean(losses)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  {phase_name} Epoch {epoch+1}: Loss={avg_loss:.4f}, LR={current_lr:.6f}, Time={elapsed:.1f}min")

        # Validate with short generation to save VRAM (keywords appear in first few tokens)
        val_acc, val_correct, val_total = evaluate(model, val_loader, desc="Validating", max_new_tokens=32)
        print(f"  Val Accuracy: {val_acc:.4f} ({val_correct}/{val_total})")

        if val_acc > best_state['best_acc']:
            best_state['best_acc'] = val_acc
            best_state['projection'] = copy.deepcopy(model.eeg_projection.state_dict())
            os.makedirs(SAVE_DIR, exist_ok=True)
            proj_path = os.path.join(SAVE_DIR, "best_projection.pth")
            torch.save(best_state['projection'], proj_path)
            model.llm.save_pretrained(os.path.join(SAVE_DIR, "best_lora"))
            print(f"  New best! Saved to {SAVE_DIR}")

        torch.cuda.empty_cache()  # Clean up after each epoch

    return best_state

In [11]:
# Separate parameters
projection_params = list(model.eeg_projection.parameters())
lora_params = [p for n, p in model.llm.named_parameters() if p.requires_grad]

print(f"Projection params: {sum(p.numel() for p in projection_params):,}")
print(f"LoRA params: {sum(p.numel() for p in lora_params):,}")

best_state = {'best_acc': 0, 'projection': None}

Projection params: 4,608,000
LoRA params: 1,126,400


In [12]:
# ---- Phase 1: Projection warmup ----
print("=" * 60)
print(f"Phase 1: Projection warmup ({WARMUP_EPOCHS} epochs)")
print("=" * 60)

# Freeze LoRA, train only projection
for p in lora_params:
    p.requires_grad = False

best_state = train_phase(
    model, train_loader, val_loader,
    trainable_params=projection_params,
    epochs=WARMUP_EPOCHS,
    lr=LR * 2.5,  # Higher LR for warmup
    phase_name="Warmup",
    grad_accum_steps=GRAD_ACCUM_STEPS,
    best_state=best_state,
)

Phase 1: Projection warmup (5 epochs)


Warmup Epoch 1/5:   0%|          | 0/696 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\Users\manoj\Projects\Mtech_Project2\CSBrain\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Warmup Epoch 1/5: 100%|██████████| 696/696 [05:05<00:00,  2.28it/s]


  Warmup Epoch 1: Loss=0.7926, LR=0.000452, Time=5.1min


Validating: 100%|██████████| 288/288 [09:31<00:00,  1.99s/it]


  Val Accuracy: 0.2769 (319/1152)
  New best! Saved to pth_downtasks/eeg_llm_bcic


...

  Val Accuracy: 0.2648 (305/1152)


Warmup Epoch 5/5: 100%|██████████| 696/696 [08:06<00:00,  1.43it/s]


  Warmup Epoch 5: Loss=0.0500, LR=0.000001, Time=8.1min


Validating: 100%|██████████| 288/288 [24:40<00:00,  5.14s/it]  


  Val Accuracy: 0.2830 (326/1152)
  New best! Saved to pth_downtasks/eeg_llm_bcic


In [13]:
# ---- Phase 2: Joint projection + LoRA training ----
joint_epochs = EPOCHS - WARMUP_EPOCHS
print("=" * 60)
print(f"Phase 2: Joint projection + LoRA training ({joint_epochs} epochs)")
print("=" * 60)

# Unfreeze LoRA
for p in lora_params:
    p.requires_grad = True

best_state = train_phase(
    model, train_loader, val_loader,
    trainable_params=projection_params + lora_params,
    epochs=joint_epochs,
    lr=LR,
    phase_name="Joint",
    grad_accum_steps=GRAD_ACCUM_STEPS,
    best_state=best_state,
)

Phase 2: Joint projection + LoRA training (15 epochs)


Joint Epoch 1/15: 100%|██████████| 696/696 [03:56<00:00,  2.95it/s]


  Joint Epoch 1: Loss=0.0566, LR=0.000198, Time=3.9min


Validating: 100%|██████████| 288/288 [08:50<00:00,  1.84s/it]


  Val Accuracy: 0.2561 (295/1152)


...

  Val Accuracy: 0.2561 (295/1152)


Joint Epoch 15/15: 100%|██████████| 696/696 [06:37<00:00,  1.75it/s]


  Joint Epoch 15: Loss=0.0361, LR=0.000001, Time=6.9min


Validating: 100%|██████████| 288/288 [12:34<00:00,  2.62s/it]

  Val Accuracy: 0.2561 (295/1152)


## 7. Test Evaluation

Load the best model and generate text for the test set.

In [15]:
# Load best projection weights
if best_state['projection'] is not None:
    model.eeg_projection.load_state_dict(best_state['projection'])
    print("Loaded best projection weights")

torch.cuda.empty_cache()  # Free VRAM before test generation
test_acc, test_correct, test_total = evaluate(model, test_loader, desc="Testing", max_new_tokens=32)
print(f"Test Accuracy (keyword extraction): {test_acc:.4f} ({test_correct}/{test_total})")

Loaded best projection weights


Testing: 100%|██████████| 288/288 [45:28<00:00,  9.47s/it]   

Test Accuracy (keyword extraction): 0.3134 (361/1152)


## 8. Sample Outputs

Generate text for a few test samples to inspect the quality of the generated descriptions.

In [17]:
model.eval()
torch.cuda.empty_cache()  # Free VRAM before sample generation
sample_batch = next(iter(test_loader))

generated_texts = model.generate(
    eeg_data=sample_batch['eeg_data'].cuda(),
    prompt_ids=sample_batch['prompt_ids'].cuda(),
    prompt_mask=sample_batch['prompt_mask'].cuda(),
    max_new_tokens=64,  # Reduced from 128 to avoid OOM; enough for sample inspection
)

label_ids = sample_batch['label_ids'].numpy()

for i, (text, true_label) in enumerate(zip(generated_texts, label_ids)):
    pred_label = extract_label(text, EMOTION_KEYWORDS)
    match = "CORRECT" if pred_label == true_label else "WRONG"
    print(f"--- Sample {i+1} [{match}] ---")
    print(f"True: {CLASS_NAMES[true_label]} (class {true_label})")
    print(f"Pred: {CLASS_NAMES.get(pred_label, 'Unknown')} (class {pred_label})")
    print(f"Generated text:{text[:300]}...")
    print()

--- Sample 1 [WRONG] ---
True: Left Hand (class 0)
Pred: Both Feet (class 2)
Generated text:The EEG recording shows motor imagery patterns for both feet movement. There is bilateral event-related desynchronization (ERD) over the central midline area (Cz), consistent with supplementary motor area activation for lower limb imagery. Midline lateralization of sensorimotor...

--- Sample 2 [WRONG] ---
True: Right Hand (class 1)
Pred: Tongue (class 3)
Generated text:Analysis of the EEG indicates tongue motor imagery. Widespread sensorimotor desynchronization with bilateral distribution suggests the subject is imagining tongue movement, activating the lateral portions of the primary motor cortex. In terms of territory, lateral portions of the motor cortex...

--- Sample 3 [CORRECT] ---
True: Right Hand (class 1)
Pred: Right Hand (class 1)
Generated text:The EEG recording shows motor imagery patterns for right hand movement. There is a clear event-related desynchronization (ERD) in the mu and 